In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install optuna
!pip install kagglehub
!pip install cuml-cu12
!pip install optuna-integration

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 10.2 MB/s eta 0:00:00


Tuning and Training Ridge Regression algorithm

In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import time
import joblib
import kagglehub

path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
            f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
            f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
            f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
            f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
            f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
            f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
            f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"}


first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Ridge Regression/Ridge_Regression_tuning.db"

all_datasets_results = []

for file in datasets:
    dataset_name = datasets[file]
    df = pd.read_parquet(file)
    df = df.rename(columns={df.columns[1]:"date"})
    df = df[df["year"].isin([2023, 2024])]
    #df["year"] = df["year"].map({2023: 0, 2024: 1})
    df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
    df = df.drop(columns=["precipitation"], errors='ignore')
    df = df.drop(columns=["total_amount"], errors="ignore")
    df = df.dropna()

    df_test = df[df["date"]>=test_set_start].copy()
    df_dev = df[df["date"]<test_set_start].copy()

    X_test = df_test.drop(columns=["trip_count","date","year"])
    y_test = df_test["trip_count"]

    X_dev = df_dev.drop(columns=["trip_count","date","year"])
    y_dev = df_dev["trip_count"]

    custom_folds = []
    for month in range(6):
        val_month_start = first_block_end + pd.DateOffset(months=month)
        val_month_end = val_month_start + pd.DateOffset(months=1)

        train_month_start = val_month_start - pd.DateOffset(months=6)

        train_start_i = df_dev["date"].searchsorted(train_month_start)
        train_end_i = df_dev["date"].searchsorted(val_month_start)
        val_end_i = df_dev["date"].searchsorted(val_month_end)

        train_indices = np.arange(train_start_i,train_end_i)
        val_indices = np.arange(train_end_i,val_end_i)

        custom_folds.append((train_indices,val_indices))

    def objective(trial):
        alpha = trial.suggest_float("alpha",1e-7,1e3,log=True) #0.0000001 to 1000

        model = Ridge(alpha=alpha)

        fold_errors = []

        for train_i, val_i in custom_folds:
            X_train, X_val = X_dev.iloc[train_i], X_dev.iloc[val_i]
            y_train, y_val = y_dev.iloc[train_i], y_dev.iloc[val_i]

            model.fit(X_train, y_train)
            pred = model.predict(X_val)

            mae = mean_absolute_error(y_val,pred)
            fold_errors.append(mae)

        return np.mean(fold_errors)


    study = optuna.create_study(
        study_name=f"ridge_{dataset_name}",
        storage=DataBase_URL,
        load_if_exists=True,
        direction="minimize"
    )

    tuning_start = time.perf_counter()
    study.optimize(objective, n_trials=80)
    tuning_time = time.perf_counter() - tuning_start

    best_alpha = study.best_params["alpha"]

    print(f"Tuning hyperparameters for {dataset_name}:{best_alpha}")

    final_model = Ridge(alpha=best_alpha)
    training_start = time.perf_counter()
    final_model.fit(X_dev, y_dev)
    training_time = time.perf_counter() - training_start

    joblib.dump(final_model, f"drive/MyDrive/BT_Florian_2026/Trained Models/Ridge Regression/{dataset_name}_model.pkl")

    testing_start = time.perf_counter()
    test_preds = final_model.predict(X_test)
    final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    final_test_mae = mean_absolute_error(y_test,test_preds)
    testing_time = time.perf_counter() - testing_start

    result_dict = {
        "Dataset": [dataset_name],
        "RMSE": [final_test_rmse],
        "MAE": [final_test_mae],
        "Tuning_Time_sec": [tuning_time],
        "Training_Time_sec": [training_time],
        "Testing_Time_sec": [testing_time]
    }

    all_datasets_results.append(result_dict)


df_final_results = pd.DataFrame(all_datasets_results)
df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Ridge Regression/RidgeRegressionResults.csv", index=False)

Tuning and Training Random Forrest Regression algorithm

In [ ]:
import pandas as pd
import numpy as np
import optuna
from cuml.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import time
import joblib
import kagglehub

path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
            f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
            f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
            f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
            f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
            f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
            f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
            f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"}


first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Random Forrest/Random_Forrest_tuning.db"

all_datasets_results = []

for file in datasets:
    dataset_name = datasets[file]
    df = pd.read_parquet(file)
    df = df.rename(columns={df.columns[1]:"date"})
    df = df[df["year"].isin([2023, 2024])]
    df["year"] = df["year"].map({2023: 0, 2024: 1})
    df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
    df = df.drop(columns=["precipitation"], errors='ignore')
    df = df.drop(columns=["total_amount"], errors="ignore")
    df = df.dropna()

    df_test = df[df["date"]>=test_set_start].copy()
    df_dev = df[df["date"]<test_set_start].copy()

    X_test = df_test.drop(columns=["trip_count","date"])
    y_test = df_test["trip_count"]

    X_dev = df_dev.drop(columns=["trip_count","date"])
    y_dev = df_dev["trip_count"]

    custom_folds = []
    for month in range(6):
        val_month_start = first_block_end + pd.DateOffset(months=month)
        val_month_end = val_month_start + pd.DateOffset(months=1)

        train_month_start = val_month_start - pd.DateOffset(months=6)

        train_start_i = df_dev["date"].searchsorted(train_month_start)
        train_end_i = df_dev["date"].searchsorted(val_month_start)
        val_end_i = df_dev["date"].searchsorted(val_month_end)

        train_indices = np.arange(train_start_i,train_end_i)
        val_indices = np.arange(train_end_i,val_end_i)

        custom_folds.append((train_indices,val_indices))

    def objective(trial):
        n_estimators = trial.suggest_int("n_estimators", 25, 100)
        max_depth = trial.suggest_int("max_depth", 10, 25)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 100, 1000, log=True)
        max_features = trial.suggest_float("max_features", 0.25, 0.75)
        max_samples = trial.suggest_float("max_samples", 0.1, 0.5)

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            max_samples=max_samples,
            random_state=100
        )

        fold_errors = []

        for train_i, val_i in custom_folds:
            X_train, X_val = X_dev.iloc[train_i], X_dev.iloc[val_i]
            y_train, y_val = y_dev.iloc[train_i], y_dev.iloc[val_i]

            model.fit(X_train, y_train)
            pred = model.predict(X_val)

            mae = mean_absolute_error(y_val,pred)
            fold_errors.append(mae)

        return np.mean(fold_errors)

    study = optuna.create_study(
        study_name=f"RF_{dataset_name}",
        storage=DataBase_URL,
        load_if_exists=True,
        direction="minimize"
    )

    tuning_start = time.perf_counter()
    study.optimize(objective, n_trials=50)
    tuning_time = time.perf_counter() - tuning_start

    best_params = study.best_params

    print(f"Best hyperparameters for {dataset_name}: {best_params}")

    final_model = RandomForestRegressor(
        **best_params,
        random_state=100
    )
    training_start = time.perf_counter()
    final_model.fit(X_dev, y_dev)
    training_time = time.perf_counter() - training_start

    joblib.dump(final_model, f"drive/MyDrive/BT_Florian_2026/Trained Models/Random Forrest/{dataset_name}_model.pkl")

    testing_start = time.perf_counter()
    test_preds = final_model.predict(X_test)
    final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    final_test_mae = mean_absolute_error(y_test,test_preds)
    testing_time = time.perf_counter() - testing_start

    result_dict = {
        "Dataset": [dataset_name],
        "RMSE": [final_test_rmse],
        "MAE": [final_test_mae],
        "Tuning_Time_sec": [tuning_time],
        "Training_Time_sec": [training_time],
        "Testing_Time_sec": [testing_time]
    }

    all_datasets_results.append(result_dict)


df_final_results = pd.DataFrame(all_datasets_results)
df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Random Forrest/RandomForrestResults.csv", index=False)

Tuning and Training Feed-Forward Neural Network models

In [ ]:
import pandas as pd
import numpy as np
import time
import optuna
from optuna_integration import TFKerasPruningCallback
import kagglehub
import tensorflow as tf
from tensorflow.keras import mixed_precision
from sklearn.metrics import mean_absolute_error, mean_squared_error

mixed_precision.set_global_policy("mixed_float16")

path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
            f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
            f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
            f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
            f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
            f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
            f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
            f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"}


first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/FFNN_tuning.db"

all_datasets_results = []

for file in datasets:
  dataset_name = datasets[file]
  df = pd.read_parquet(file)
  df = df.rename(columns={df.columns[1]:"date"})
  df = df[df["year"].isin([2023, 2024])]
  df["year"] = df["year"].map({2023: 0, 2024: 1})
  df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
  df = df.drop(columns=["precipitation"], errors='ignore')
  df = df.drop(columns=["total_amount"], errors="ignore")
  df = df.dropna()

  df_test = df[df["date"]>=test_set_start].copy()
  df_dev = df[df["date"]<test_set_start].copy()

  X_test = df_test.drop(columns=["trip_count","date"])
  y_test = df_test["trip_count"]

  X_dev = df_dev.drop(columns=["trip_count","date"])
  y_dev = df_dev["trip_count"]

  X_dev = tf.constant(X_dev.values, dtype=tf.float32)
  y_dev = tf.constant(y_dev.values, dtype=tf.float32)

  custom_folds = []
  for month in range(6):
    val_month_start = first_block_end + pd.DateOffset(months=month)
    val_month_end = val_month_start + pd.DateOffset(months=1)

    train_month_start = val_month_start - pd.DateOffset(months=6)

    train_start_i = df_dev["date"].searchsorted(train_month_start)
    train_end_i = df_dev["date"].searchsorted(val_month_start)
    val_end_i = df_dev["date"].searchsorted(val_month_end)

    train_indices = np.arange(train_start_i,train_end_i)
    val_indices = np.arange(train_end_i,val_end_i)

    custom_folds.append((train_indices,val_indices))

  def objective(trial):
    hidden_layers = trial.suggest_int("hidden_layers", 1, 3)
    neurons_per_layer = trial.suggest_categorical("neurons_per_layer", [8, 16, 32, 64, 128])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [2048, 4096, 8192, 16384, 32768])

    fold_epochs = []
    fold_errors = []

    for fold_id, (train_i, val_i) in enumerate(custom_folds):
      tf.keras.backend.clear_session()

      X_train, X_val = tf.gather(X_dev, train_i), tf.gather(X_dev, val_i)
      y_train, y_val = tf.gather(y_dev, train_i), tf.gather(y_dev, val_i)

      model = tf.keras.Sequential()
      model.add(tf.keras.Input(shape=(X_train.shape[1],)))

      for i in range(hidden_layers):
        model.add(tf.keras.layers.Dense(neurons_per_layer, activation="relu"))

      model.add(tf.keras.layers.Dense(1, activation="linear"))

      model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="mae")

      early_stopper = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
      callbacks = [early_stopper]
      if fold_id == 0:
        callbacks.append(TFKerasPruningCallback(trial, "val_loss"))

      history = model.fit(
          X_train, y_train,
          validation_data=(X_val, y_val),
          batch_size=batch_size,
          epochs=100,
          callbacks=callbacks,
          verbose=0
      )

      best_epoch = np.argmin(history.history["val_loss"]) + 1
      fold_epochs.append(best_epoch)

      mae = model.evaluate(X_val, y_val, verbose=0)
      fold_errors.append(mae)

    trial.set_user_attr("optimal_epochs", round(np.mean(fold_epochs)))

    return np.mean(fold_errors)

  ram_storage = optuna.storages.InMemoryStorage()

  study = optuna.create_study(
      study_name=f"NN_{dataset_name}",
      storage=ram_storage,
      load_if_exists=True,
      direction="minimize",
      pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
  )

  tuning_start = time.perf_counter()
  study.optimize(objective, n_trials=3)
  tuning_time = time.perf_counter() - tuning_start

  optuna.copy_study(
      from_study_name=f"NN_{dataset_name}",
      from_storage=ram_storage,
      to_storage=DataBase_URL,
      to_study_name=f"NN_{dataset_name}"
  )

  best_params = study.best_params

  avg_best_epochs = study.best_trial.user_attrs["optimal_epochs"]
  best_params["epochs"] = int(np.round(avg_best_epochs))

  tf.keras.backend.clear_session()

  final_model = tf.keras.Sequential()
  final_model.add(tf.keras.Input(shape=(X_dev.shape[1],)))

  for i in range(best_params["hidden_layers"]):
    final_model.add(tf.keras.layers.Dense(
        best_params["neurons_per_layer"],
        activation="relu"
    ))

  final_model.add(tf.keras.layers.Dense(1, activation="linear"))

  final_model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["learning_rate"]),
      loss="mae"
  )

  training_start = time.perf_counter()

  history = final_model.fit(
      X_dev, y_dev,
      batch_size=best_params["batch_size"],
      epochs=best_params["epochs"],
      verbose=0
  )

  training_time = time.perf_counter() - training_start

  final_model.save(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/{dataset_name}_model.keras")

  testing_start = time.perf_counter()

  test_preds = final_model.predict(X_test)

  test_preds = test_preds.flatten()

  final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
  final_test_mae = mean_absolute_error(y_test, test_preds)

  testing_time = time.perf_counter() - testing_start

  history_df = pd.DataFrame(history.history)
  history_df.to_csv(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/{dataset_name}_history.csv", index=False)

  result_dict = {
      "Dataset": [dataset_name],
      "RMSE": [final_test_rmse],
      "MAE": [final_test_mae],
      "Tuning_Time_sec": [tuning_time],
      "Training_Time_sec": [training_time],
      "Testing_Time_sec": [testing_time]
  }

  all_datasets_results.append(result_dict)

df_final_results = pd.DataFrame(all_datasets_results)
df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/NeuralNetworkResults.csv", index=False)


True
1
Using Colab cache for faster access to the 'nyc-taxi-data-2023-24-normalized' dataset.


[I 2026-08-19 17:53:45,388] A new study created in memory with name: NN_df_DOs
[I 2026-08-19 17:57:55,440] Trial 0 finished with value: 0.4273512214422226 and parameters: {'hidden_layers': 2, 'neurons_per_layer': 8, 'learning_rate': 0.0011427010875829197, 'batch_size': 8192}. Best is trial 0 with value: 0.4273512214422226.
[I 2026-08-19 18:01:59,563] Trial 1 finished with value: 0.3657824496428172 and parameters: {'hidden_layers': 3, 'neurons_per_layer': 32, 'learning_rate': 0.000425000489887486, 'batch_size': 16384}. Best is trial 1 with value: 0.3657824496428172.
[I 2026-08-19 18:05:44,041] Trial 2 finished with value: 0.4824635088443756 and parameters: {'hidden_layers': 1, 'neurons_per_layer': 64, 'learning_rate': 5.1091566484555386e-05, 'batch_size': 32768}. Best is trial 1 with value: 0.3657824496428172.
[I 2026-08-19 18:05:44,356] A new study created in RDB with name: NN_df_DOs


69629/69629 ━━━━━━━━━━━━━━━━━━━━ 84s 1ms/step


[I 2026-08-19 18:08:49,374] A new study created in memory with name: NN_df_DOsWEATHER
[I 2026-08-19 18:11:51,993] Trial 0 finished with value: 0.23448706914981207 and parameters: {'hidden_layers': 3, 'neurons_per_layer': 128, 'learning_rate': 0.0060800091369413965, 'batch_size': 8192}. Best is trial 0 with value: 0.23448706914981207.
[I 2026-08-19 18:20:12,790] Trial 1 finished with value: 0.2382267788052559 and parameters: {'hidden_layers': 3, 'neurons_per_layer': 128, 'learning_rate': 0.0007590374005555742, 'batch_size': 2048}. Best is trial 0 with value: 0.23448706914981207.
[I 2026-08-19 18:24:04,150] Trial 2 finished with value: 0.4621707499027252 and parameters: {'hidden_layers': 1, 'neurons_per_layer': 8, 'learning_rate': 0.001840253811701709, 'batch_size': 2048}. Best is trial 0 with value: 0.23448706914981207.
[I 2026-08-19 18:24:04,212] A new study created in RDB with name: NN_df_DOsWEATHER


69629/69629 ━━━━━━━━━━━━━━━━━━━━ 86s 1ms/step


...

69061/69061 ━━━━━━━━━━━━━━━━━━━━ 86s 1ms/step


[I 2026-08-19 20:02:02,761] A new study created in memory with name: NN_df_PUsCYCLt
[I 2026-08-19 20:04:45,894] Trial 0 finished with value: 0.2981708844502767 and parameters: {'hidden_layers': 3, 'neurons_per_layer': 64, 'learning_rate': 0.007015233677109573, 'batch_size': 32768}. Best is trial 0 with value: 0.2981708844502767.
[I 2026-08-19 20:08:32,758] Trial 1 finished with value: 0.30129555116097134 and parameters: {'hidden_layers': 2, 'neurons_per_layer': 128, 'learning_rate': 0.00842861776573202, 'batch_size': 2048}. Best is trial 0 with value: 0.2981708844502767.
[I 2026-08-19 20:14:25,770] Trial 2 finished with value: 0.4345170557498932 and parameters: {'hidden_layers': 2, 'neurons_per_layer': 8, 'learning_rate': 0.00013100688577417162, 'batch_size': 8192}. Best is trial 0 with value: 0.2981708844502767.
[I 2026-08-19 20:14:25,837] A new study created in RDB with name: NN_df_PUsCYCLt


69061/69061 ━━━━━━━━━━━━━━━━━━━━ 85s 1ms/step


[I 2026-08-19 20:17:05,227] A new study created in memory with name: NN_df_PUsWEATHER_CYCLt
[I 2026-08-19 20:19:43,828] Trial 0 finished with value: 0.41280707716941833 and parameters: {'hidden_layers': 1, 'neurons_per_layer': 32, 'learning_rate': 0.003325880730973871, 'batch_size': 16384}. Best is trial 0 with value: 0.41280707716941833.
[I 2026-08-19 20:23:38,788] Trial 1 finished with value: 0.4494486004114151 and parameters: {'hidden_layers': 3, 'neurons_per_layer': 8, 'learning_rate': 0.0002167933108877314, 'batch_size': 32768}. Best is trial 0 with value: 0.41280707716941833.
[I 2026-08-19 20:30:09,656] Trial 2 finished with value: 0.3520021140575409 and parameters: {'hidden_layers': 3, 'neurons_per_layer': 64, 'learning_rate': 0.0001429270704012304, 'batch_size': 4096}. Best is trial 2 with value: 0.3520021140575409.
[I 2026-08-19 20:30:09,720] A new study created in RDB with name: NN_df_PUsWEATHER_CYCLt


69061/69061 ━━━━━━━━━━━━━━━━━━━━ 85s 1ms/step


In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import time
import joblib
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
            f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
            f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
            f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
            f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
            f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
            f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
            f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"}


first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/FFNN_tuning.db"

all_datasets_results = []

for file in datasets:
    dataset_name = datasets[file]
    df = pd.read_parquet(file)
    df = df.rename(columns={df.columns[1]:"date"})
    df = df[df["year"].isin([2023, 2024])]
    df["year"] = df["year"].map({2023: 0, 2024: 1})
    df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
    df = df.drop(columns=["precipitation"], errors='ignore')
    df = df.drop(columns=["total_amount"], errors="ignore")
    df = df.dropna()

    df_test = df[df["date"]>=test_set_start].copy()
    df_dev = df[df["date"]<test_set_start].copy()

    X_test = df_test.drop(columns=["trip_count","date"])
    y_test = df_test["trip_count"]

    X_dev = df_dev.drop(columns=["trip_count","date"])
    y_dev = df_dev["trip_count"]

    custom_folds = []
    for month in range(6):
        val_month_start = first_block_end + pd.DateOffset(months=month)
        val_month_end = val_month_start + pd.DateOffset(months=1)

        train_month_start = val_month_start - pd.DateOffset(months=6)

        train_start_i = df_dev["date"].searchsorted(train_month_start)
        train_end_i = df_dev["date"].searchsorted(val_month_start)
        val_end_i = df_dev["date"].searchsorted(val_month_end)

        train_indices = np.arange(train_start_i,train_end_i)
        val_indices = np.arange(train_end_i,val_end_i)

        custom_folds.append((train_indices,val_indices))

    def objective(trial):
        hidden_layers = trial.suggest_int("hidden_layers",1,5)
        neurons_per_layer = trial.suggest_categorical("neurons_per_layer", [16, 32, 64, 128, 256])
        hidden_layer_sizes = tuple([neurons_per_layer] * hidden_layers)

        learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True)
        alpha = trial.suggest_float("alpha", 1e-4, 1e-1, log=True)

        batch_size = trial.suggest_categorical("batch_size", [1024, 2048, 4096, 8192, 16384])

        model = MLPRegressor(
            hidden_layer_sizes=hidden_layer_sizes,
            learning_rate_init=learning_rate_init,
            alpha=alpha,
            batch_size=batch_size,
            activation="relu",
            max_iter=500,
            early_stopping=True,
            random_state=100
        )

        fold_errors = []

        for train_i, val_i in custom_folds:
            X_train, X_val = X_dev.iloc[train_i], X_dev.iloc[val_i]
            y_train, y_val = y_dev.iloc[train_i], y_dev.iloc[val_i]

            model.fit(X_train, y_train)
            pred = model.predict(X_val)

            mae = mean_absolute_error(y_val,pred)
            fold_errors.append(mae)

        return np.mean(fold_errors)

    study = optuna.create_study(
        study_name=f"NN_{dataset_name}",
        storage=DataBase_URL,
        load_if_exists=True,
        direction="minimize"
    )

    tuning_start = time.perf_counter()
    study.optimize(objective, n_trials=50)
    tuning_time = time.perf_counter() - tuning_start

    best_params = study.best_params

    print(f"Best hyperparameters for {dataset_name}: {best_params}")

    hidden_layers = best_params.pop("hidden_layers")
    neurons = best_params.pop("neurons_per_layer")

    best_params["hidden_layer_sizes"] = tuple([neurons] * hidden_layers)

    final_model = MLPRegressor(
        **best_params,
        activation="relu",
        max_iter=500,
        early_stopping=True,
        random_state=100
    )
    training_start = time.perf_counter()
    final_model.fit(X_dev, y_dev)
    training_time = time.perf_counter() - training_start

    joblib.dump(final_model, f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/{dataset_name}_model.pkl")

    testing_start = time.perf_counter()
    test_preds = final_model.predict(X_test)
    final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    final_test_mae = mean_absolute_error(y_test,test_preds)
    testing_time = time.perf_counter() - testing_start

    result_dict = {
        "Dataset": [dataset_name],
        "RMSE": [final_test_rmse],
        "MAE": [final_test_mae],
        "Tuning_Time_sec": [tuning_time],
        "Training_Time_sec": [training_time],
        "Testing_Time_sec": [testing_time]
    }

    all_datasets_results.append(result_dict)


df_final_results = pd.DataFrame(all_datasets_results)
df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/NeuralNetworkResults.csv", index=False)

Using Colab cache for faster access to the 'nyc-taxi-data-2023-24-normalized' dataset.


[I 2026-08-16 11:17:26,105] Using an existing study with name 'NN_df_DOs' instead of creating a new one.
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
import pandas as pd
import numpy as np
import time
import optuna
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class FFNN(nn.Module):
  def __init__(self, input_dim, hidden_layers, neurons_per_layer):
    super(FFNN, self).__init__()
    layers = []
    in_features = input_dim

    for _ in range(hidden_layers):
      layers.append(nn.Linear(in_features, neurons_per_layer))
      layers.append(nn.ReLU())
      in_features = neurons_per_layer

    layers.append(nn.Linear(in_features, 1))

    self.network = nn.Sequential(*layers)

  def forward(self, x):
    return self.network(x).squeeze()

def train_model(model, X_df, y_series, batch_size, lr, alpha, epochs):
    X_tensor = torch.tensor(X_df.values, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_series.values, dtype=torch.float32).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=alpha)

    model.to(device)
    dataset_size = len(X_tensor)

    for epoch in range(epochs):
        model.train()

        indices = torch.randperm(dataset_size, device=device)

        for i in range(0, dataset_size, batch_size):
            batch_idx = indices[i : i + batch_size]
            batch_X = X_tensor[batch_idx]
            batch_y = y_tensor[batch_idx]

            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()


def predict_model(model, X_df, batch_size):
    X_tensor = torch.tensor(X_df.values, dtype=torch.float32).to(device)
    dataset_size = len(X_tensor)

    model.eval()
    model.to(device)
    all_preds = []

    with torch.no_grad():
        for i in range(0, dataset_size, batch_size):
            batch_X = X_tensor[i : i + batch_size]
            preds = model(batch_X)

            all_preds.append(preds)

    return torch.cat(all_preds).cpu().numpy()


path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
            f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
            f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
            f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
            f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
            f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
            f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
            f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"}


first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:////content/FFNN_tuning.db"

all_datasets_results = []

for file in datasets:
    dataset_name = datasets[file]
    df = pd.read_parquet(file)
    df = df.rename(columns={df.columns[1]:"date"})
    df = df[df["year"].isin([2023, 2024])]
    df["year"] = df["year"].map({2023: 0, 2024: 1})
    df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
    df = df.drop(columns=["precipitation"], errors='ignore')
    df = df.drop(columns=["total_amount"], errors="ignore")
    df = df.dropna()

    df_test = df[df["date"]>=test_set_start].copy()
    df_dev = df[df["date"]<test_set_start].copy()

    X_test = df_test.drop(columns=["trip_count","date"])
    y_test = df_test["trip_count"]

    X_dev = df_dev.drop(columns=["trip_count","date"])
    y_dev = df_dev["trip_count"]

    input_dim = X_dev.shape[1]

    custom_folds = []
    for month in range(6):
        val_month_start = first_block_end + pd.DateOffset(months=month)
        val_month_end = val_month_start + pd.DateOffset(months=1)
        train_month_start = val_month_start - pd.DateOffset(months=6)

        train_start_i = df_dev["date"].searchsorted(train_month_start)
        train_end_i = df_dev["date"].searchsorted(val_month_start)
        val_end_i = df_dev["date"].searchsorted(val_month_end)

        train_indices = np.arange(train_start_i,train_end_i)
        val_indices = np.arange(train_end_i,val_end_i)

        custom_folds.append((train_indices,val_indices))

    def objective(trial):
        hidden_layers = trial.suggest_int("hidden_layers", 1, 5)
        neurons_per_layer = trial.suggest_categorical("neurons_per_layer", [16, 32, 64, 128, 256])
        learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True)
        alpha = trial.suggest_float("alpha", 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical("batch_size", [1024, 2048, 4096, 8192, 16384])
        epochs = trial.suggest_int("epochs", 30, 150)

        fold_errors = []

        for train_i, val_i in custom_folds:
            X_train, X_val = X_dev.iloc[train_i], X_dev.iloc[val_i]
            y_train, y_val = y_dev.iloc[train_i], y_dev.iloc[val_i]

            model = FFNN(input_dim=input_dim, hidden_layers=hidden_layers, neurons_per_layer=neurons_per_layer)

            train_model(
                model=model,
                X_df=X_train,
                y_series=y_train,
                batch_size=batch_size,
                lr=learning_rate_init,
                alpha=alpha,
                epochs=epochs
            )

            pred = predict_model(model, X_val, batch_size=batch_size)

            mae = mean_absolute_error(y_val, pred)
            fold_errors.append(mae)

        return np.mean(fold_errors)

    study = optuna.create_study(
        study_name=f"NN_{dataset_name}",
        storage=DataBase_URL,
        load_if_exists=True,
        direction="minimize"
    )

    tuning_start = time.perf_counter()
    study.optimize(objective, n_trials=30)
    tuning_time = time.perf_counter() - tuning_start

    best_params = study.best_params
    print(f"Best hyperparameters for {dataset_name}: {best_params}")

    final_model = FFNN(
        input_dim=input_dim,
        hidden_layers=best_params["hidden_layers"],
        neurons_per_layer=best_params["neurons_per_layer"]
    )

    training_start = time.perf_counter()
    train_model(
        model=final_model,
        X_df=X_dev,
        y_series=y_dev,
        batch_size=best_params["batch_size"],
        lr=best_params["learning_rate_init"],
        alpha=best_params["alpha"],
        epochs=best_params["epochs"]
    )
    training_time = time.perf_counter() - training_start

    torch.save(final_model.state_dict(), f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/{dataset_name}_model.pth")

    testing_start = time.perf_counter()
    test_preds = predict_model(final_model, X_test, batch_size=best_params["batch_size"])
    final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    final_test_mae = mean_absolute_error(y_test, test_preds)
    testing_time = time.perf_counter() - testing_start

    result_dict = {
      "Dataset": [dataset_name],
      "RMSE": [final_test_rmse],
      "MAE": [final_test_mae],
      "Tuning_Time_sec": [tuning_time],
      "Training_Time_sec": [training_time],
      "Testing_Time_sec": [testing_time]
    }

    all_datasets_results.append(result_dict)

df_final_results = pd.DataFrame(all_datasets_results)
df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/NeuralNetworkResults.csv", index=False)
!cp /content/FFNN_tuning.db "/drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/FFNN_tuning.db"


In [ ]:
import pandas as pd
import numpy as np
import time
import optuna
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error

class FFNN(nn.Module):
  def __init__(self, input_size, hidden_layers, neurons_per_layer):
    super(FFNN, self).__init__()
    layers = []
    input = input_size
    for i in range(hidden_layers):
      layers.append(nn.Linear(input, neurons_per_layer))
      layers.append(nn.ReLU())
      input = neurons_per_layer
    layers.append(nn.Linear(input, 1))
    self.network = nn.Sequential(*layers)

  def forward(self,x):
    return self.network(x).squeeze()


path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
            f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
            f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
            f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
            f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
            f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
            f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
            f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"}

first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:///FFNN_tuning.db"

all_datasets_results = []

for file in datasets:
  dataset_name = datasets[file]
  df = pd.read_parquet(file)
  df = df.rename(columns={df.columns[1]:"date"})
  df = df[df["year"].isin([2023, 2024])]
  df["year"] = df["year"].map({2023: 0, 2024: 1})
  df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
  df = df.drop(columns=["precipitation"], errors='ignore')
  df = df.drop(columns=["total_amount"], errors="ignore")
  df = df.dropna()

  df_test = df[df["date"]>=test_set_start].copy()
  df_dev = df[df["date"]<test_set_start].copy()

  X_test = df_test.drop(columns=["trip_count","date"])
  y_test = df_test["trip_count"]

  X_dev = df_dev.drop(columns=["trip_count","date"])
  y_dev = df_dev["trip_count"]

  custom_folds = []
  for month in range(6):
    val_month_start = first_block_end + pd.DateOffset(months=month)
    val_month_end = val_month_start + pd.DateOffset(months=1)

    train_month_start = val_month_start - pd.DateOffset(months=6)

    train_start_i = df_dev["date"].searchsorted(train_month_start)
    train_end_i = df_dev["date"].searchsorted(val_month_start)
    val_end_i = df_dev["date"].searchsorted(val_month_end)

    train_indices = np.arange(train_start_i,train_end_i)
    val_indices = np.arange(train_end_i,val_end_i)

    custom_folds.append((train_indices,val_indices))

  def objective(trial):
    hidden_layers = trial.suggest_int("hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_categorical("neurons_per_layer", [16, 32, 64, 128, 256])
    learning_rate = trial.suggest_int("learning_rate", 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [1024, 2048, 4096, 8192, 16384])

    model = FFNN(input_size=X_dev.shape[1], hidden_layers=hidden_layers, neurons_per_layer=neurons_per_layer)
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    fold_errors = []

    for train_i, val_i in custom_folds:
      X_train, X_val = X_dev.iloc[train_i], X_dev.iloc[val_i]
      y_train, y_val = y_dev.iloc[train_i], y_dev.iloc[val_i]

      train_data = TensorDataset(X_train, y_train)
      val_data = TensorDataset(X_val, y_val)

      train_loader = DataLoader(train_data, batch_size=batch_size)
      val_loader = DataLoader(val_data, batch_size=batch_size)

      patience_max = 3
      patience = 0
      best_val_loss = float("inf")

      for epoch in range(100):
        model.train()

        for X_batch, y_batch in train_loader:
          optimizer.zero_grad()
          outputs = model(X_batch)
          loss = criterion(outputs, y_batch)
          loss.backward()
          optimizer.step()
          train_loss += loss.item()

        model.eval()
        val_loss = 0

        with torch.no_grad():
          for X_batch, y_batch in val_loader:
              outputs = model(X_batch)
              loss = criterion(outputs, y_batch)
              val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        if avg_val_loss < best_val_loss:
          best_val_loss = avg_val_loss
          patience = 0
        else:
          patience += 1
          if patience >= patience_max:
            break


